# 08 · XGBoost Deep Dive — servability & the head-to-head

**Phase goal:** close the loop on the model bake-off (nb 05). XGBoost had the *best* cross-validated MAE, yet the pipeline shipped **LightGBM**. This notebook answers the two questions that leaves open:

1. *Why* couldn't XGBoost be shipped through a scikit-learn `Pipeline`?
2. If we serve it another way, **is XGBoost actually better** than the LightGBM we shipped — i.e., did the selection cost us any accuracy?

Full write-up: [`docs/XGBOOST_SERVABILITY.md`](../docs/XGBOOST_SERVABILITY.md).

In [ ]:
import sys, warnings
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

from car_pricing import config, data, features, models
from car_pricing.pipeline import make_pipeline

df = data.clean(data.load_raw())
X, y = features.split_xy(df)

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=config.TEST_SIZE, random_state=config.RANDOM_STATE)
edges = features.band_edges(ytr)
print('train/test:', len(Xtr), '/', len(Xte))

train/test: 15856 / 3964


## 1 · Reproduce the failure
XGBoost fits inside a `Pipeline` fine — the break is at **`.predict()`**, when scikit-learn 1.6 inspects the estimator's tags.

In [2]:
# Fitting through the Pipeline works fine; the break is at .predict()
pipe = make_pipeline(models.model_zoo()['XGBoost'])
pipe.fit(Xtr, ytr)          # fitting is fine

try:
    pipe.predict(Xte[:3])
    print('predict OK (env already patched)')
except Exception as e:
    print('predict FAILS ->', type(e).__name__ + ':', str(e)[:70])

predict FAILS -> AttributeError: 'super' object has no attribute '__sklearn_tags__'


## 2 · Root cause — a broken cooperative-inheritance chain
scikit-learn 1.6 builds estimator tags by walking the MRO and calling `super().__sklearn_tags__()` cooperatively. That requires the **mixin first, `BaseEstimator` last**. XGBoost 2.1.1 declares the bases the other way round, so `RegressorMixin`'s `super()` lands on `object` — which has no `__sklearn_tags__` → `AttributeError`.

In [3]:
# Inspect the method-resolution order (MRO) that triggers the bug
mro = [c.__name__ for c in XGBRegressor.__mro__]
print('XGBRegressor MRO:', ' -> '.join(mro))
print('BaseEstimator before RegressorMixin?',
      mro.index('BaseEstimator') < mro.index('RegressorMixin'),
      '(that ordering is the bug)')

import xgboost, sklearn
print('xgboost', xgboost.__version__, '| scikit-learn', sklearn.__version__)

XGBRegressor MRO: XGBRegressor -> XGBModel -> BaseEstimator -> _HTMLDocumentationLinkMixin -> _MetadataRequester -> RegressorMixin -> object
BaseEstimator before RegressorMixin? True (that ordering is the bug)
xgboost 2.1.1 | scikit-learn 1.6.1


### Fix options
| Option | What | Verdict |
| :-- | :-- | :-- |
| **Upgrade** | `pip install 'xgboost>=2.1.2'` (or pin `scikit-learn<1.6`) | ✅ Cleanest — XGBoost then drops straight into the Pipeline |
| **Decoupled serving** | Fit the `ColumnTransformer` + `XGBRegressor` separately; `xgb.predict(pre.transform(X))` | ✅ Works on any versions (used below to evaluate) |
| Monkey-patch tags | Inject a `__sklearn_tags__` at runtime | ⚠️ Fragile / version-specific — not recommended |

## 3 · Serve XGBoost decoupled and evaluate on the held-out test set

In [4]:
# Decoupled serving: fit the preprocessor and the regressor SEPARATELY, then
# predict on pre.transform(X). This sidesteps the Pipeline tag bug entirely and
# works on any library versions.
pre = features.build_preprocessor()
Xtr_e = pre.fit_transform(Xtr, ytr)
Xte_e = pre.transform(Xte)

xgb = XGBRegressor(**models.model_zoo()['XGBoost'].get_params())
xgb.fit(Xtr_e, ytr)
pred = xgb.predict(Xte_e)


def score(yt, p):
    """Return R2 / MAE / RMSE plus the derived band accuracy."""
    tb = features.price_to_band(yt, edges)
    pb = features.price_to_band(p, edges)
    return {'R2': round(r2_score(yt, p), 4),
            'MAE': round(mean_absolute_error(yt, p), 3),
            'RMSE': round(float(np.sqrt(mean_squared_error(yt, p))), 3),
            'band_acc': round(float((tb == pb).mean()), 4)}


print('XGBoost (decoupled) TEST:', score(yte, pred))

XGBoost (decoupled) TEST: {'R2': 0.9568, 'MAE': 0.668, 'RMSE': 1.021, 'band_acc': 0.8605}


## 4 · Head-to-head: XGBoost vs the shipped LightGBM

In [ ]:
import json

comp = json.loads(config.COMPARISON_PATH.read_text())
m = json.loads(config.METRICS_PATH.read_text())
xgb_test = score(yte, pred)

tbl = pd.DataFrame({
    'XGBoost': {
        'CV MAE': round(comp['XGBoost']['cv_mae'], 3),
        'Test MAE': xgb_test['MAE'],
        'Test R2': xgb_test['R2'],
        'Test RMSE': xgb_test['RMSE'],
        'Band acc': xgb_test['band_acc'],
        'Servable in Pipeline': 'no (this env)',
    },
    'LightGBM (shipped)': {
        'CV MAE': round(comp['LightGBM']['cv_mae'], 3),
        'Test MAE': m['price_mae_lakhs'],
        'Test R2': m['price_r2'],
        'Test RMSE': m['price_rmse_lakhs'],
        'Band acc': m['band_accuracy'],
        'Servable in Pipeline': 'yes',
    },
})

tbl

,XGBoost,LightGBM (shipped)
CV MAE,0.708,0.715
Test MAE,0.668,0.664
Test R2,0.9568,0.9568
Test RMSE,1.021,1.022
Band acc,0.8605,0.8582
Servable in Pipeline,no (this env),yes


## 5 · Fine-tuning XGBoost — the knobs that matter

Sections 1–4 answered *servability*. This section answers the other half of "is XGBoost any good?": **how do you tune it?** XGBoost exposes a lot of hyperparameters, but they fall into four intuitive groups — you rarely touch more than these.

### The four groups of knobs

| Group | Hyperparameter | What it controls | Turning it **up** → |
| :-- | :-- | :-- | :-- |
| **Learning schedule** | `learning_rate` (η) | Step size — how strongly each new tree corrects the last | Faster but rougher fit; can overshoot the optimum |
| | `n_estimators` | Number of boosting rounds (trees) | More capacity; slower; eventually overfits |
| **Tree complexity** | `max_depth` | How deep each tree grows = order of feature interactions | Captures richer interactions; overfits |
| | `min_child_weight` | Min. summed instance weight required in a leaf | **Simpler** trees (stronger regularisation) |
| | `gamma` (min_split_loss) | Min. loss reduction needed to make a split | **Simpler** trees (stronger regularisation) |
| **Randomness (bagging)** | `subsample` | Fraction of **rows** sampled per tree | Less overfitting up to a point; too low underfits |
| | `colsample_bytree` | Fraction of **columns** sampled per tree | Decorrelates trees; too low underfits |
| **Regularisation** | `reg_lambda` (L2) | Penalty on the size of leaf weights | Smoother, more conservative model |
| | `reg_alpha` (L1) | Sparsity penalty on leaf weights | Pushes some leaf weights to exactly zero |

**The single most important relationship:** `learning_rate` and `n_estimators` trade off against each other. A *smaller* learning rate needs *more* trees to reach the same fit, but usually generalises better. The standard recipe is therefore: pick a small learning rate (≈0.03–0.1), then let **early stopping** choose `n_estimators` for you.

In [ ]:
# The XGBoost hyperparameters this project ships (from car_pricing.models.model_zoo)
shipped = models.model_zoo()['XGBoost'].get_params()
for k in ['n_estimators', 'max_depth', 'learning_rate', 'subsample', 'colsample_bytree']:
    print(f'{k:18s} {shipped[k]}')

### How to actually tune: search, don't guess

Two workhorse strategies, both leakage-safe because they score every candidate with **cross-validation**:

- **`RandomizedSearchCV`** — sample random combinations from ranges you specify. The best bang-for-buck default: it explores a large space far more efficiently than an exhaustive grid when several knobs matter at once.
- **`GridSearchCV`** — try *every* combination on a small, hand-picked grid. Use it only to *refine* one or two parameters once a random search has put you in the right neighbourhood.

The demo below runs a small randomized search over the key knobs. It is deliberately tiny (`n_iter=20`, `cv=3`) so it finishes quickly — widen the ranges and raise `n_iter` for a real search. It reuses the pre-encoded matrices `Xtr_e` / `Xte_e` from section 3.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

# uniform(loc, scale) samples in [loc, loc + scale]
search_space = {
    'n_estimators':     randint(300, 900),      # number of boosting rounds
    'max_depth':        randint(3, 9),          # interaction depth per tree
    'learning_rate':    uniform(0.02, 0.10),    # shrinkage: 0.02 .. 0.12
    'subsample':        uniform(0.70, 0.30),    # row sampling:    0.70 .. 1.00
    'colsample_bytree': uniform(0.70, 0.30),    # column sampling: 0.70 .. 1.00
    'min_child_weight': randint(1, 8),          # larger => simpler trees
    'reg_lambda':       uniform(0.0, 3.0),      # L2 penalty on leaf weights
}

base = XGBRegressor(objective='reg:squarederror', tree_method='hist',
                    n_jobs=-1, random_state=config.RANDOM_STATE, verbosity=0)

search = RandomizedSearchCV(
    base, search_space, n_iter=20, cv=3,
    scoring='neg_mean_absolute_error',   # CV MAE, in Lakhs (higher = better here)
    random_state=config.RANDOM_STATE, n_jobs=-1,
)
search.fit(Xtr_e, ytr)

print('best CV MAE :', round(-search.best_score_, 3), 'Lakhs')
print('best params :')
for k, v in sorted(search.best_params_.items()):
    print(f'   {k:18s} {round(v, 4) if isinstance(v, float) else v}')

### Let the data pick `n_estimators`: early stopping

Rather than guessing the tree count, give XGBoost far more rounds than you need, hold out a small validation slice of the **training** set, and stop as soon as the validation error fails to improve for `early_stopping_rounds` consecutive rounds. This is the cleanest way to settle the learning-rate ↔ `n_estimators` pair: **fix a small learning rate, and let early stopping trim the tree count automatically.**

In [ ]:
# Carve a validation slice out of the (already encoded) training set
Xfit, Xval, yfit, yval = train_test_split(
    Xtr_e, ytr, test_size=0.2, random_state=config.RANDOM_STATE)

tuned = XGBRegressor(
    n_estimators=3000,          # a generous upper bound; early stopping trims it
    learning_rate=0.03,         # small LR + many rounds => a finer, smoother fit
    max_depth=6,
    subsample=0.9,
    colsample_bytree=0.9,
    min_child_weight=3,
    reg_lambda=1.0,
    early_stopping_rounds=50,   # stop after 50 rounds with no val improvement
    eval_metric='mae',
    tree_method='hist',
    n_jobs=-1, random_state=config.RANDOM_STATE, verbosity=0,
)
tuned.fit(Xfit, yfit, eval_set=[(Xval, yval)], verbose=False)

print('early stopping chose n_estimators =', tuned.best_iteration + 1)
print('tuned TEST :', score(yte, tuned.predict(Xte_e)))

### A practical tuning recipe

1. **Start sane.** `learning_rate=0.1`, `max_depth=6`, `subsample=0.9`, `colsample_bytree=0.9` — close to what this project ships.
2. **Set `n_estimators` by early stopping** with a validation slice (above), never by hand.
3. **Tune complexity next:** search `max_depth` (3–10) and `min_child_weight` (1–10) *together* — they jointly govern overfitting.
4. **Add stochasticity:** tune `subsample` and `colsample_bytree` in 0.6–1.0 to decorrelate the trees.
5. **Add regularisation** (`reg_lambda`, `reg_alpha`, `gamma`) only if the model still overfits.
6. **Lower the learning rate** for the final model (e.g. 0.1 → 0.03) and re-run early stopping — a small, reliable accuracy gain for a longer training time.

> **Why this project doesn't ship a heavily tuned XGBoost:** the bake-off (nb 05) fixes *modest, comparable* hyperparameters across every model family so the comparison is fair — the goal there is model *selection*, not squeezing out the last 0.1%. The recipe above is what you would run *after* selection to squeeze a chosen model, and — per sections 1–2 — you would still need `xgboost>=2.1.2` to serve the result through the standard Pipeline.

## Conclusion
- **The models are statistically indistinguishable on the held-out test set** — identical R², MAE within ~₹400, band accuracy within 0.2 pt. Note LightGBM is even *marginally better on test MAE* despite XGBoost's slightly better CV MAE, confirming the gap is noise.
- **Shipping LightGBM therefore cost zero accuracy** while buying clean, shim-free serialisation and serving — exactly the operational-robustness trade-off `train.py` automates.
- **To ship XGBoost instead**, upgrade to `xgboost>=2.1.2`; it then drops straight into the same `Pipeline` and the selector would pick it automatically. Until then, decoupled serving (above) is the escape hatch.

The bake-off thread is now fully closed.